# UAS Pengembangan Teknologi Ucapan (PTU)
## Pipeline Pelatihan Model STT (Wav2Vec2) & TTS (SpeechT5) Kustom dengan Suara Sendiri

Notebook ini berisi alur lengkap untuk melatih kedua model yang disyaratkan dalam UAS:
1. **Wav2Vec2** (untuk Speech-to-Text / STT)
2. **SpeechT5** (untuk Text-to-Speech / TTS)

Skrip ini dirancang untuk dijalankan di **Google Colab (dengan T4 GPU)**.

## TAHAP 1: Instalasi Dependensi & Persiapan Environment

In [ ]:
# Menginstal pustaka yang diperlukan untuk ASR (Wav2Vec2) dan TTS (SpeechT5)
!pip install transformers[torch] datasets accelerate evaluate torchaudio soundfile librosa speechbrain tensorboard

In [ ]:
# Import library dasar
import os
import torch
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Menghubungkan Google Drive untuk membaca dataset
drive.mount('/content/drive')

## TAHAP 2: Pembacaan Dataset & Normalisasi Audio

Pada bagian ini, pastikan Anda telah menaruh folder dataset Anda di Google Drive dengan struktur:
`/content/drive/MyDrive/my_voice_dataset/`  
di mana di dalamnya terdapat subfolder `wavs/`, file `metadata_stt.csv`, dan file `metadata_tts.txt`.

In [ ]:
# Tentukan path folder dataset di Google Drive Anda (Tanpa perlu di-zip)
DATASET_PATH = "/content/drive/MyDrive/my_voice_dataset"
WAVS_DIR = os.path.join(DATASET_PATH, "wavs")
METADATA_STT = os.path.join(DATASET_PATH, "metadata_stt.csv")
METADATA_TTS = os.path.join(DATASET_PATH, "metadata_tts.txt")

print("Memeriksa keberadaan berkas dataset...")
print("Folder wavs:", os.path.exists(WAVS_DIR))
print("Berkas metadata_stt.csv:", os.path.exists(METADATA_STT))
print("Berkas metadata_tts.txt:", os.path.exists(METADATA_TTS))

if not (os.path.exists(WAVS_DIR) and os.path.exists(METADATA_STT) and os.path.exists(METADATA_TTS)):
    raise FileNotFoundError("Dataset tidak lengkap. Pastikan path Drive Anda sudah benar.")

### 2.1 Normalisasi Audio ke 16.000 Hz (16kHz) Mono
Karena Anda merekam menggunakan perangkat mic pribadi dengan sample rate yang bervariasi, kita perlu menormalisasikan seluruh audio ke sample rate standar model (**16.000 Hz**) dan mengubahnya menjadi Mono.

In [ ]:
# Load dataset STT
df_stt = pd.read_csv(METADATA_STT)
df_stt['file_path'] = df_stt['file_name'].apply(lambda x: os.path.join(DATASET_PATH, x))

print("Memulai proses normalisasi audio ke 16kHz mono...")
for idx, row in df_stt.iterrows():
    audio_path = row['file_path']
    if os.path.exists(audio_path):
        # Load vokal dan resample paksa ke 16kHz
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        # Tulis ulang audio dengan format PCM WAV 16-bit
        sf.write(audio_path, y, 16000)
    else:
        print(f"Peringatan: Berkas {audio_path} tidak ditemukan!")

print("Seluruh berkas audio berhasil dinormalisasikan ke 16kHz mono!")

---

## TAHAP 3: Pelatihan & Evaluasi Model STT (Wav2Vec2)

In [ ]:
from datasets import Dataset, Audio

# Konversi ke HuggingFace Dataset Format
stt_dataset = Dataset.from_pandas(df_stt)
stt_dataset = stt_dataset.cast_column("file_path", Audio(sampling_rate=16000))
print(stt_dataset[0])

In [ ]:
import json
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# 1. Ekstrak karakter unik dari transkripsi untuk membuat vocabulary
def extract_all_chars(batch):
    all_text = " ".join(batch["transcription"])
    vocab = list(set(all_text))
    return {"vocab": [vocab]}

vocabs = stt_dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=stt_dataset.column_names)
vocab_list = list(set([char for vocab in vocabs["vocab"] for char in vocab]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

# Tambahkan special tokens
vocab_dict["|"] = vocab_dict.pop(" ")
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

# 2. Inisialisasi Tokenizer & Feature Extractor
tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained("./wav2vec2_processor")
print("Vocabulary & Processor Wav2Vec2 siap!")

In [ ]:
# 3. Pra-pemrosesan data untuk model
def prepare_dataset(batch):
    audio = batch["file_path"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    
    with processor.as_target_processor():
        batch["labels"] = processor(batch["transcription"]).input_ids
    return batch

stt_prepared = stt_dataset.map(prepare_dataset, remove_columns=stt_dataset.column_names)
stt_split = stt_prepared.train_test_split(test_size=0.1)
print("Dataset siap untuk dilatih!")

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Dict, List, Union

# 4. Data Collator untuk padding dinamis pada batch
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# 5. Inisialisasi model dasar Wav2Vec2 Indonesia
model = Wav2Vec2ForCTC.from_pretrained(
    "indonesian-nlp/wav2vec2-large-xlsr-indonesian",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

# Membekukan feature extractor agar tidak berubah
model.freeze_feature_extractor()

In [ ]:
# 6. Pengaturan parameter training & mulai melatih
training_args = TrainingArguments(
    output_dir="./wav2vec2_results",
    group_by_length=True,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    num_train_epochs=30,
    fp16=True,
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    learning_rate=3e-4,
    warmup_steps=50,
    save_total_limit=2,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=stt_split["train"],
    eval_dataset=stt_split["test"],
    tokenizer=processor.feature_extractor,
)

print("=== Memulai Training Wav2Vec2 ===")
trainer.train()

# 7. Simpan Model Hasil Latih
model.save_pretrained("./wav2vec2_custom")
processor.save_pretrained("./wav2vec2_custom")
print("Model Wav2Vec2 kustom Anda telah berhasil disimpan di folder: ./wav2vec2_custom")

---

## TAHAP 4: Pelatihan & Evaluasi Model TTS (SpeechT5)

In [ ]:
# 1. Load data metadata TTS
tts_data = []
with open(METADATA_TTS, 'r') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            file_id, text = parts[0], parts[1]
            tts_data.append({
                "file_path": os.path.join(WAVS_DIR, f"{file_id}.wav"),
                "normalized_text": text
            })

df_tts = pd.DataFrame(tts_data)
tts_dataset = Dataset.from_pandas(df_tts)
tts_dataset = tts_dataset.cast_column("file_path", Audio(sampling_rate=16000))
print(tts_dataset[0])

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

# 2. Ekstrak speaker embedding unik dari suara Anda menggunakan model ECAPA-TDNN
print("Memuat pengklasifikasi pembicara (SpeechBrain)... ")
classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", run_opts={"device": "cuda"})

def create_speaker_embedding(waveform):
    with torch.no_grad():
        embedding = classifier.encode_batch(torch.tensor(waveform).unsqueeze(0))
        embedding = torch.nn.functional.normalize(embedding, dim=-1)
        return embedding[0, 0].cpu().numpy()

print("Mengekstrak x-vector speaker embedding dari berkas audio...")
embeddings = []
for item in tts_dataset:
    embeddings.append(create_speaker_embedding(item["file_path"]["array"]))

# Simpan speaker embedding rata-rata sebagai representasi vokal Anda
mean_embedding = np.mean(embeddings, axis=0)
np.save("speaker_embedding.npy", mean_embedding)
print("Speaker embedding berhasil diekstrak dan disimpan ke: speaker_embedding.npy")

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

# 3. Memuat Model Dasar SpeechT5
processor_tts = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model_tts = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

def prepare_dataset_tts(batch):
    audio = batch["file_path"]
    input_ids = processor_tts(text=batch["normalized_text"], return_tensors="pt").input_ids[0]
    
    # Ekstraksi target Mel-Spectrogram
    y = audio["array"]
    S = librosa.feature.melspectrogram(y=y, sr=16000, n_fft=1024, hop_length=256, n_mels=80)
    log_mel = librosa.power_to_db(S, ref=np.max).T
    
    batch["input_ids"] = input_ids.tolist()
    batch["labels"] = log_mel.astype(np.float32).tolist()
    batch["speaker_embeddings"] = mean_embedding.tolist()
    return batch

tts_prepared = tts_dataset.map(prepare_dataset_tts, remove_columns=tts_dataset.column_names)
tts_split = tts_prepared.train_test_split(test_size=0.1)
print("Data spektrogram untuk model TTS siap!")

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from dataclasses import dataclass
from typing import Dict, List, Union

# 4. Data Collator untuk SpeechT5 (mengatur alignment & padding data log-mel)
@dataclass
class TTSDataCollatorWithPadding:
    processor: SpeechT5Processor

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        labels = [{"input_values": feature["labels"]} for feature in features]
        speaker_embeddings = [feature["speaker_embeddings"] for feature in features]

        batch = self.processor.pad(input_ids, return_tensors="pt")
        
        max_label_len = max(len(l["input_values"]) for l in labels)
        padded_labels = []
        for l in labels:
            mel = np.array(l["input_values"])
            pad_width = max_label_len - len(mel)
            padded_mel = np.pad(mel, ((0, pad_width), (0, 0)), mode='constant', constant_values=-80.0)
            padded_labels.append(padded_mel)
            
        batch["labels"] = torch.tensor(np.array(padded_labels), dtype=torch.float32)
        batch["speaker_embeddings"] = torch.tensor(speaker_embeddings, dtype=torch.float32)
        return batch

data_collator_tts = TTSDataCollatorWithPadding(processor=processor_tts)

In [ ]:
# 5. Training parameters & proses melatih SpeechT5
training_args_tts = Seq2SeqTrainingArguments(
    output_dir="./speecht5_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    num_train_epochs=50,
    save_steps=200,
    eval_steps=200,
    logging_steps=10,
    learning_rate=1e-4,
    warmup_steps=100,
    save_total_limit=2,
    fp16=True,
    predict_with_generate=True
)

trainer_tts = Seq2SeqTrainer(
    model=model_tts,
    args=training_args_tts,
    train_dataset=tts_split["train"],
    eval_dataset=tts_split["test"],
    data_collator=data_collator_tts,
    tokenizer=processor_tts.tokenizer,
)

print("=== Memulai Training SpeechT5 ===")
trainer_tts.train()

# 6. Simpan Model Hasil Latih
model_tts.save_pretrained("./speecht5_custom")
processor_tts.save_pretrained("./speecht5_custom")
print("Model SpeechT5 kustom Anda telah berhasil disimpan di folder: ./speecht5_custom")

---

## TAHAP 5: Pengarsipan & Penyimpanan Model Kembali ke Google Drive

In [ ]:
import shutil

print("Memulai pengarsipan model...")
# Copy speaker embedding npy ke dalam folder model tts kustom agar terintegrasi
shutil.copy("speaker_embedding.npy", "./speecht5_custom/speaker_embedding.npy")

# Kompres kedua folder model menjadi file ZIP
shutil.make_archive("wav2vec2_custom", "zip", "./wav2vec2_custom")
shutil.make_archive("speecht5_custom", "zip", "./speecht5_custom")

# Pindahkan file ZIP hasil kompresi kembali ke Google Drive Anda
shutil.move("wav2vec2_custom.zip", os.path.join(DATASET_PATH, "wav2vec2_custom.zip"))
shutil.move("speecht5_custom.zip", os.path.join(DATASET_PATH, "speecht5_custom.zip"))

print("=== PROSES SELESAI ===")
print(f"File zip model hasil latih Anda telah disimpan di Drive Anda pada folder:")
print(f"1. {os.path.join(DATASET_PATH, 'wav2vec2_custom.zip')}")
print(f"2. {os.path.join(DATASET_PATH, 'speecht5_custom.zip')}")
print("Silakan unduh kedua file zip tersebut ke komputer lokal Anda untuk diekstraksi ke folder model backend.")